# **Goal**

The purpose of this notebook is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

###**Pythia Model Family**

- Pythia models were originally designed for interpretability research.

- All Pythia models were trained on the same data in the same order.

- Training data: All Pythia models are trained on The Pile. The Pile is a 825GiB general-purpose dataset in English. It contains texts from 22 diverse sources, roughly broken down into five categories: academic writing (e.g. arXiv), internet (e.g. CommonCrawl), prose (e.g. Project Gutenberg), dialogue (e.g. YouTube subtitles), and miscellaneous (e.g. GitHub, Enron Emails).

- Each model saw 299,892,736,000 ~= 300B tokens during training.

####**Pythia 160M**

- <u>Total params</u>: 162,322,944. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 85,056,000. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 12
- <u>Model dimension</u>: 768. I.e.: Every token is represented internally by a vector of length 768.
- <u>MLP units per layer:</u> 3072
- <u>Heads</u>: 12
- <u>Vocabulary size:</u> 50,304. Amount of tokens in vocabulary.
- <u>Max context window</u>: 2048 tokens.

Load model

In [1]:
from transformers import GPTNeoXForCausalLM, AutoTokenizer

model = GPTNeoXForCausalLM.from_pretrained(
  "EleutherAI/pythia-160m-deduped",
  revision="step143000",
  cache_dir="./pythia-160m-deduped/step143000",
)

tokenizer = AutoTokenizer.from_pretrained(
  "EleutherAI/pythia-160m-deduped",
  revision="step143000",
  cache_dir="./pythia-160m-deduped/step143000",
)

inputs = tokenizer("Hello, I am", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  649MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'Hello, I am a newbie to the web and I have been trying to get my head around the concept of a'

Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [2]:
inputs = tokenizer("How much do you like ice cream?", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=28) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'How much do you like ice cream?\n\nI love ice cream. I love the ice cream. I love the ice cream. I'

In [3]:
inputs = tokenizer("The capital of France is", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'The capital of France is the city of Paris, and the capital of the world.\n\nThe French capital is the capital'

In [4]:

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(output[0])
print(len(output[0]))
print()
print(tokenizer.decode(output[0]))

tensor([  510,  5347,   273,  6181,   310,   253,  2846,   273,  7785,    15,
          187,   187,   510,  5347,   310,   671,   253,  4201,  5070,   273,
          253,  5685, 17502,  7924,    71, 16120, 35441,    15,   733,   369,
        11420,   407,   253,  5685, 15796, 41888, 16120, 35441,   275,   253,
         5768,   273, 26052,    15,   754,   369,   247,  8530,   285, 20803,
         4677,   275,  6176,    15,   187])
55

The capital of France is the city of Paris.

The capital is also the birthplace of the German poet Josef Selbst. It was founded by the German soldier Ernst Selbst in the summer of 1918. He was a famous and influential figure in Germany.



In [5]:
inputs = tokenizer("What is love?", return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

Inputs:tensor([1276,  310, 2389,   32])
Outputs: tensor([  187, 26617,   390,  9239,    13,   984,   352,  2789,   368,  1928,
         8872,   285, 25106,    15, 10540,   417,   281,   320,  3815,   275,
          667,  1039,  1425,   313,    49,  1714,  7980,    10,   773, 23337,
          247,  1436,   556,   271,  1774,  1659,   387,   253,  2798,   273,
         3253,   359,   513,    28,   697,  1318,   476,  1620,  1551,   327])

What is love?
love or hate, because it makes you feel sad and lonely. Love not to be alone in any way.” (Pope Francis) “Love a person has an important place at the heart of everything we do; its value can never rest on


Here we are just printing the internal shape of one layer of the model.

In [6]:
print(model.gpt_neox.layers[0].mlp)

GPTNeoXMLP(
  (dense_h_to_4h): Linear(in_features=768, out_features=3072, bias=True)
  (dense_4h_to_h): Linear(in_features=3072, out_features=768, bias=True)
  (act): GELUActivation()
)


In [7]:
model.config.intermediate_size

3072

The input vector representation is 768-dimensional. The number of MLP units in the layer is 3072. Each of these produces a scalar from the input through learned weights and bises, so the layer produces a transformed and expanded 3072-d vector. After activation using GELU (Gaussian Error Linear Unit), these values are transformed back into a new 768-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{GELU}(x) = x \cdot \Phi(x) = 0.5x \left(1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right)$, where $Φ(x)$ is the cumulative distribution function of the standard normal distribution, and $\text{erf}(x)$ is the Gauss error function.

**MLP Unit Ablation Experiments**

In [8]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # Load weights in layer layer_idx
    mlp = model.gpt_neox.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    # Indexes in [0, 3071]
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    #
    def ablation_hook(module, inputs, output):
        indices_device = indices.to(output.device)
        output = output.clone()
        output[..., indices_device] = 0
        return output

    handle = mlp.act.register_forward_hook(ablation_hook)

    return handle, indices, layer_idx

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [9]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,  2846,    13,   285,   697,  3072,   556,   644,  3629,  1580,
         1283,  2941,    15,   187,   688,  1722,  2511,   352,   369,   247,
         1659,   835,  1142,   952,   497,  3811,   275, 14192,    28,  3063,
          627,   403,   625,   685,   337,  3041, 21251,   665,  3153,  2708,
          616,  2097,    27,   597,   452,   642, 16347,   390,  6152,  8364])

Zeroed activations in 0.0% of units in layer 0
The capital of France is the city, and its population has been increasing since 1803.
In 1789 it was a place where many people were living in poverty; today there are more than 1 million inhabitants who live below their means: they have no savings or credit cards


$25\%$ of first layer

In [10]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,   954,  1774,   285,    13,   187,  1171,   512,    15,   380,
         5112,   403,   247,  5674,   326,   556,   644,   275,  6242,   323,
          625,   685,   767, 14020,    28,   352,   369, 11420,   407, 32436,
         6490,   327,  2552,   337,   296,  1722,  4590,   313,   395,   534,
          344,   574,  2168,  1160,   521,   806,  3177,   387, 14275,   432])

Zeroed activations in 25.0% of units in layer 0
The capital of France is the most important and,
of all. The French are a nation that has been in existence for more than two centuries; it was founded by Napoleon III on May 1st 1793 (and which he had already made his first attempt at independence from


$50\%$ of first layer

In [11]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 253,  954, 1774,  285,   13,  275,  512, 2892,   15,  187,  380, 5112,
        1342,  369,  247,  637,  665,  556,  644,  271, 9558,  326,  344,  434,
         417,  581,  281,  320,  521, 1445,  390,  352,  347,  973,  323,  436,
         309,  452,  594, 1199,  625,  685,  368,  403,  619, 1495,  533,  752,
         476,  359])

Zeroed activations in 50.0% of units in layer 0
The capital of France is the most important and, in all history.
 The Frenchman was a man who has been an artist that he's not one to be his art or it as well for this I have so much more than you are my life but what can we


$75\%$ of first layer

In [12]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  187,    13,    15,    28,     3,   407,    14,    27,    16,   285,
            2,   275,   253,  1051,   281,   432,   247, 50276,   186, 50275,
        50274,   209,   623, 28910, 50272,    29, 50273,   996, 50269,   551,
        50271,    31, 50267,    33,  2490,  1040,   313,   535,  1214,   387,
          544,   342,   327,    62,   340,   745,  3706,  6781,  4725,   748])

Zeroed activations in 75.0% of units in layer 0
The capital of France is
,.;" by-:/ and! in the... to from a  	        		 	      <     
	          {       >           @ 
 | (

 @ at [ with on] y off ; --> /> }


$100\%$ of first layer

In [13]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/3072*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  187,   423,    13,    15,    27,    28,     2,   407,    14,   544,
          367,   275,   268,   591,  3402,  2245,  2177,  4980,   327,  7503,
         1612,    62,   362,   295,  1708,   285,    16,  4376,   246,  6152,
          278,  9176,  1386, 12275,   941,  1048,  3148,  2978,  1979,  1755,
          418,   673,   186,   996,   313,     3,    10,    31,    60,   535])

Zeroed activations in 100.0% of units in layer 0
The capital of France is
end,.:;! by- [ P in p per wall port vi modern on virtual power] v n light and/ price t credit m pin line pixel data long display length size top L time	
	 (")>[




**Attention Head Ablation Experiments**

First let's check the attention heads

In [14]:
print(model.gpt_neox.layers[0].attention)

GPTNeoXAttention(
  (query_key_value): Linear(in_features=768, out_features=2304, bias=True)
  (dense): Linear(in_features=768, out_features=768, bias=True)
)


We see that the attention mechanism expects a 768d input (the token embedding or output from the previous block), which gets turned into a 2304-d vector. This corresponds to calculating its query, key, and value vectors and concatenating these. There are 12 heads computing the attention mechanisms between QKV in slices of 64d vectors.

These attention contributions from each head are then concatenated to form another 768d vector, which in turn gets transformed into the final attention contribution vector that combines and mixes the computations from all heads, and is incorporated into the input for the next transformer block.

In [15]:
print(model.gpt_neox.layers[0].attention.dense)

Linear(in_features=768, out_features=768, bias=True)


In [16]:
model.config.hidden_size

768

In [17]:
model.config.num_attention_heads

12

In [18]:
model.config.hidden_size // model.config.num_attention_heads

64

Attention head ablation function

In [19]:
def add_head_ablation(model, head_indices, layer_idx):

    num_layers = model.config.num_hidden_layers
    #Embedding dimension
    hidden = model.config.hidden_size
    #Number of attention heads
    num_heads = model.config.num_attention_heads
    #Projection dimension per attention head
    head_dim = hidden // num_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")

    # Load final attention module
    attention_dense = model.gpt_neox.layers[layer_idx].attention.dense

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            if not 0 <= head_idx < num_heads:
                raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    # Note this is a pre-hook because we want to intervene components before the forward pass.
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [21]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,  6253,   285,   954,  1774,    15,   187,   688,  1635,   281,
          436,    13,   352,   556,   247,  1180,   275,   697,  1211,   987,
           27,   380,  5112,   403,   581,    14, 25512,   394,   313,    18,
           16,    20,    10,   347,  1142,   952,  3153,   627,   685,   667,
          643,  2586,   327,  6149,    28,   597,   452,   625,  2583,   323])

Zeroed head [0] in layer 0
The capital of France is the largest and most important.
In addition to this, it has a number in its own right: The French are one-fifth (1/3) as many people live there than any other country on earth; they have more money for


Ablate 3 attention heads in layer 0

---



In [22]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,   954,  1774,   285,  6253,    15,   187,   688,  1635,   281,
         1146,   247,  2201,  2846,    13,   352,   556,   671,   644,   581,
          323,  1142,   643,  4343,   275,  3060,   347,   973,    27,   380,
        16333,   313,   746,   520,   582,  6176,     9,  1093,  1525,    10,
          390, 21753,    14, 38288,   552,    16, 16796,  5182,   428, 29171])

Zeroed head [0, 1, 2] in layer 0
The capital of France is the most important and largest.
In addition to being a major city, it has also been one for many other countries in Europe as well: The Netherlands (1901), Germany(1899) or Austria-Hungary/Austria - Hungary


Ablate 6 attention heads in layer 0

In [23]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,  6253,  2586,   275,  3060,    15,   187,   783,  1533,   434,
          954,  1684, 15873,  5674,    13,   285,   352,   556,   247,  3072,
          326,  3797,   625,   685,   337,  3041,   952,   432,   512,   689,
         3729,  3968,    28,   627,   403,   670,   374,  6494,  7108,  3811,
         1060,  3063,   313,   395,   597,   452,   644,   323,  8007,   481])

Zeroed head [0, 1, 2, 3, 4, 5] in layer 0
The capital of France is the largest country in Europe.
the world's most populous nation, and it has a population that includes more than 1 million people from all over North America; there are about 2 billion Americans living here today (and they have been for decades).


Ablate 9 attention heads in layer 0

In [24]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5,7,8,9],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 253, 6253, 2586,  275, 3060,   15,  187,  395,  352,  556,  247, 3072,
         326,  434,  670,  337,   13,  933,  952,  285,  697, 6982,  369, 1077,
        2266,  387,  436,  673,  449,  313, 2413, 1358, 2700,   18,   14,   17,
          89,   19,   10,  380, 5962, 2846,  327, 6149,  342,  271, 2170, 4409,
         625,  685])

Zeroed head [0, 1, 2, 3, 4, 5, 7, 8, 9] in layer 0
The capital of France is the largest country in Europe.
and it has a population that's about 1,000 people and its economy was very strong at this time." (http://www1-0x2) The biggest city on earth with an area worth more than


Ablate 12 attention heads in layer 0

In [25]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5,6,7,8,9,10,11],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253, 17572,  4683,   380,  3645,   187,    36,  2198,   695,    83,
          610,    84,  2967,  2676,  1378,    85,  1440,    77,   620, 33685,
           79,     3,    53,  4490,  2293,  1476,     2,   346,    42,   308,
        26909,   418,  4985,    48,  3271,   449,   473, 24580,    15,    46,
         2277,    40,  8643,    41,    13,   309,  1353,   417,   247,  1943])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] in layer 0
The capital of France is the Capital Of The World
Cristricrrysrsrrrttttllllllln"TTTLL!"! "I T TT LOOOOL." OMG.MOMGMMH, I'm not a big


Ablate 12 attention heads in layer 11

In [26]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5,6,7,8,9,10,11],
    layer_idx=11,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,  2846,   326,   556,   644,   247,  2201,  4760,   275,  3060,
          323,   625,   685,   767,  8007,    15,   187,   688,  1635,   281,
          697,  6349,   347,   271,  1774,  2245,    13,   352,   671,  3400,
         2289,   285,  5454, 15050,   875,  1142,  5284,  4343,  1690,  6176,
          313, 37796,   434,  6253, 11947,  7832,   582,  9972,    28, 11268])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] in layer 11
The capital of France is the city that has been a major player in Europe for more than two decades.
In addition to its importance as an important port, it also provides access and trade routes between many European countries including Germany (Germany's largest trading partner), Italy; Spain


Ablate heads in every layer

Baseline

In [40]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Baseline")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Baseline
The capital of France is the city, and its population has been increasing since 1803.
In 1789 it was a place where many people were living in poverty; today there are more than 1 million inhabitants who live below their means: they have no savings or credit cards


In [33]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed head 0 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed head 0 in all layers
The capital of France is the home to its it, and.
It's a place where you can be yourself in your life as well-being!Q:

  How do I get an array from JSON? 
    var json = [{"id":1,"


In [34]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2 in all layers
The capital of France is the it, which (that(the that.
  it's a) to be its own name and not in any other way than "I" or even if you don't like this one more then I can say my favorite song from your


In [36]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2,3,4,5] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5 in all layers
The capital of France is the state.
Ted to beware, and a lot more than one-to make it's possible for them all over again! The other people in their home town or country where they are not only because I have been able with this year


In [37]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2,3,4,5,6,7,8] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7,8 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5,6,7,8 in all layers
The capital of France is it
Гュージェールされたりますのでしていない場合にはあるものになることなくてられているようならしいですべるときにしたいんじゃろうかったためだけれ qu


In [38]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2,3,4,5,6,7,8,9,10,11] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7,8,9,10,11 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5,6,7,8,9,10,11 in all layers
The capital of France is a)
therapy. OEMS, and the same as well-based on their own an example: "normalized by defaulted to beacrossing that they arealivey in this time periodicality for instance when it
